# Laboratorio 11. Redes neuronales

## MLP con Sklearn

In [ ]:
from sklearn.datasets import load_iris

data = load_iris()
print(data.DESCR)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

X = data.data
y = data.target

print(X.shape, y.shape, np.unique(y))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=123)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(64,32),
    activation="relu",
    alpha=0.0001,
    max_iter=500,
    random_state=123
)
mlp

In [ ]:
mlp.fit(X_train, y_train)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(mlp.loss_curve_)
plt.xlabel("Epocas")
plt.ylabel("Funcion de perdida")
plt.grid()
plt.show()


## MLP con TensorFlow

https://archive.ics.uci.edu/dataset/186/wine+quality

In [ ]:
pip install ucimlrepo

In [ ]:
print(wine_quality.data.keys())

In [ ]:
wine_quality.data.original

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
wine_quality = fetch_ucirepo(id=186)

# data (as pandas dataframes)

df = wine_quality.data.original

red_wine = df[df["color"] == "red"]

X = red_wine.drop(columns=["color", "quality"])
y = red_wine["quality"]

In [ ]:
X.info()

In [ ]:
import matplotlib.pyplot as plt

freq = y.value_counts()

plt.bar(freq.index, freq.values)
plt.show()

In [ ]:
X.shape, y.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=123)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

In [ ]:
import tensorflow as tf

regression_model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],), batch_size=32),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="linear")
])

regression_model.summary()

In [ ]:
regression_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=[
        tf.keras.metrics.R2Score(),
        tf.keras.metrics.MeanAbsoluteError(),
    ]
)

regression_model.fit(X_train_scaled, y_train, epochs=100, validation_split=0.2)

In [ ]:
regression_model.history.history.keys()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,6))

ax[0].plot(regression_model.history.history["loss"], label="Train")
ax[0].plot(regression_model.history.history["val_loss"], label="Validation")
ax[0].set_ylabel("LOSS")

ax[1].plot(regression_model.history.history["r2_score"], label="Train")
ax[1].plot(regression_model.history.history["val_r2_score"], label="Validation")
ax[1].set_ylabel("$R^2$")

for a in ax:
    a.set_xlabel("Epocas")
    a.grid(c="0.85")
    a.legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=123
)

from sklearn.preprocessing import StandardScaler, OneHotEncoder

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
y_train_transformed = y_train - y.min()

In [ ]:
import tensorflow as tf

classification_model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],), batch_size=32),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(7, activation="softmax")
])

classification_model.summary()

In [ ]:
classification_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy()
    ]
)

classification_model.fit(X_train_scaled, y_train_transformed, epochs=100, validation_split=0.2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,6))

ax[0].plot(classification_model.history.history["loss"], label="Train")
ax[0].plot(classification_model.history.history["val_loss"], label="Validation")
ax[0].set_ylabel("LOSS")

ax[1].plot(classification_model.history.history["sparse_categorical_accuracy"], label="Train")
ax[1].plot(classification_model.history.history["val_sparse_categorical_accuracy"], label="Validation")
ax[1].set_ylabel("$R^2$")

for a in ax:
    a.set_xlabel("Epocas")
    a.grid(c="0.85")
    a.legend()

plt.tight_layout()
plt.show()